# Phase 3 Overview Notebook: RoBERTa Sentiment & Pretrained Sarcasm Profiling

This notebook provides an interactive environment to inspect and validate Phase 3 v2 sentiment scoring:
1. **Primary Sentiment Engine**: CardiffNLP Twitter-RoBERTa (`cardiffnlp/twitter-roberta-base-sentiment-latest`).
2. **Pretrained Sarcasm Classifier**: CardiffNLP Twitter-Irony (`cardiffnlp/twitter-roberta-base-irony`).
3. **VADER Baseline Comparison**: Continuous VADER compound scoring.
4. **Model Agreement Diagnostics**: Pearson correlation ($r$) and label agreement matrix.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
PROJECT_ROOT = Path("..").resolve()
P3_DATA_DIR = PROJECT_ROOT / "data" / "02_interim" / "phase3_v2"
P3_RESULTS_DIR = PROJECT_ROOT / "output" / "results" / "phase3" / "v2"

manifest = json.loads((P3_RESULTS_DIR / "sentiment_manifest_v2.json").read_text())
print("Phase 3 Manifest Summary:")
print(json.dumps(manifest, indent=2))

## 1. Load Scored Dataset (`twitter_sentiment_v2.parquet`)

In [ ]:
sentiment_df = pd.read_parquet(P3_DATA_DIR / "twitter_sentiment_v2.parquet")
print(f"Total Scored Tweets: {len(sentiment_df):,}")
sentiment_df[["tweet_cleaned", "roberta_score", "roberta_label", "vader_compound", "vader_label", "sarcasm_risk_score"]].head(5)

## 2. Compare RoBERTa vs VADER Score Distributions

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(sentiment_df["roberta_score"], label="RoBERTa Sentiment Score", color="#3977C3", fill=True, alpha=0.3)
sns.kdeplot(sentiment_df["vader_compound"], label="VADER Compound Score", color="#E69F00", fill=True, alpha=0.3)
plt.title("Density Comparison: RoBERTa vs. VADER Continuous Sentiment Scores", fontsize=14, fontweight="bold")
plt.xlabel("Sentiment Valence [-1.0, +1.0]", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Pretrained Sarcasm Risk Score Distribution

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(sentiment_df["sarcasm_risk_score"], bins=50, kde=True, color="#8E44AD")
plt.title(f"Pretrained Irony/Sarcasm Risk Score Distribution (Mean: {sentiment_df['sarcasm_risk_score'].mean():.4f})", fontsize=14, fontweight="bold")
plt.xlabel("Sarcasm Risk Probability [0.0, 1.0]", fontsize=12)
plt.ylabel("Tweet Count", fontsize=12)
plt.tight_layout()
plt.show()